# DPO: beta, reference drift and noisy labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/direct_preference_optimisation.ipynb)

Self-contained CPU experiment. Install missing dependencies with `%pip install torch numpy scipy matplotlib`. Retain Colab's supplied PyTorch build. Tested on Python 3.12, PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development-machine GPU build.

In [ ]:
import math, time
import numpy as np
import torch
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
torch.set_num_threads(4)
from torch import nn

## Sequence logp

The mask specifies response tokens after causal alignment. Excluded logits are replaced before softmax, so even a padding row of -inf logits contributes zero safely. This test does not determine whether the caller selected the correct response positions.

In [ ]:
def sequence_logp(logits, tokens, mask):
    included = mask.bool()
    safe_logits = logits.masked_fill(~included.unsqueeze(-1), 0.0)
    token_logp = safe_logits.log_softmax(-1).gather(-1, tokens.unsqueeze(-1)).squeeze(-1)
    return token_logp.masked_fill(~included, 0.0).sum(-1)

## Dpo loss

Compute the chosen-minus-rejected policy log-ratio relative to the frozen reference, scale by beta, then apply the stable negative log-sigmoid.

In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta):
    margin = policy_chosen - policy_rejected - (ref_chosen - ref_rejected)
    return -F.logsigmoid(beta * margin).mean()

## Preference data

Twelve known contexts have four categorical responses. Labels are noisy Bradley-Terry draws from fixed rewards. The evaluation has fresh labels on the same contexts, so it measures preference fitting rather than unseen-context generalisation.

In [ ]:
def preference_data(seed, n=4096):
    g = torch.Generator().manual_seed(seed)
    ctx = torch.randint(12, (n,), generator=g)
    a = torch.randint(4, (n,), generator=g)
    b = (a + torch.randint(1, 4, (n,), generator=g)) % 4
    rewards = torch.tensor([[1.2 * np.sin(c * 0.7 + j * 1.3) + 0.5 * np.cos(c + j) for j in range(4)] for c in range(12)])
    first = torch.rand(n, generator=g) < torch.sigmoid(rewards[ctx, a] - rewards[ctx, b])
    chosen = torch.where(first, a, b)
    rejected = torch.where(first, b, a)
    return (ctx, chosen, rejected, rewards)

## Dpo experiment

Check the exact nonzero-margin gradient, categorical initial loss, excluded reference parameters and masked -inf rows. Adam trains three beta values across three minibatch seeds. Population policies and label-noise ceilings explain the expected comparison.

In [ ]:
def dpo_experiment():
    start = time.perf_counter()
    ctx, chosen_id, rejected_id, rewards = preference_data(120)
    test_ctx, test_chosen, test_rejected, _ = preference_data(999, 4096)
    reference_log_probs = torch.log_softmax(torch.tensor([[0.6 * np.cos(c + j * 0.8) for j in range(4)] for c in range(12)]), -1)
    initial_logits = reference_log_probs.clone().requires_grad_(True)
    initial_policy = initial_logits.log_softmax(-1)
    initial = float(dpo_loss(initial_policy[ctx, chosen_id], initial_policy[ctx, rejected_id], reference_log_probs[ctx, chosen_id], reference_log_probs[ctx, rejected_id], 0.5).detach())
    assert abs(initial - math.log(2)) < 1e-06
    margin = torch.tensor(-2.0, dtype=torch.float64, requires_grad=True)
    # A nonzero margin catches beta placement as well as the gradient sign.
    checked_beta = 0.5
    checked_loss = dpo_loss(margin, torch.tensor(0.0), torch.tensor(0.0), torch.tensor(0.0), checked_beta)
    checked_loss.backward()
    expected_gradient = -checked_beta * torch.sigmoid(-checked_beta * margin.detach())
    assert torch.allclose(margin.grad, expected_gradient, atol=1e-12)
    mask_logits = torch.randn(2, 3, 4)
    tokens = torch.tensor([[1, 2, 0], [3, 1, 2]])
    mask = torch.tensor([[1, 1, 0], [1, 0, 0]])
    before = sequence_logp(mask_logits, tokens, mask)
    changed = mask_logits.clone()
    changed[mask == 0] = -torch.inf
    assert torch.allclose(before, sequence_logp(changed, tokens, mask))
    # This ordering is Bayes optimal for the generator; finite labels add noise.
    realised_ceiling = float((rewards[test_ctx, test_chosen] > rewards[test_ctx, test_rejected]).float().mean())
    expected_ceiling = float(torch.stack([torch.sigmoid(abs(rewards[c, a] - rewards[c, b])) for c in range(12) for a in range(4) for b in range(a + 1, 4)]).mean())
    # Population targets separate objective behaviour from finite Adam training.
    records, traces, targets = ([], [], [])
    for beta in [0.1, 0.5, 1.0]:
        optimal = (reference_log_probs + rewards / beta).log_softmax(-1)
        targets.append(dict(beta=beta, kl=float((optimal.exp() * (optimal - reference_log_probs)).sum(-1).mean()), probabilities=optimal.exp().tolist()))
    for seed in [0, 1, 2]:
        for beta in [0.1, 0.5, 1.0]:
            torch.manual_seed(seed)
            logits = nn.Parameter(reference_log_probs.clone())
            fixed_reference = reference_log_probs.detach().clone()
            opt = torch.optim.Adam([logits], lr=0.03)
            assert not fixed_reference.requires_grad
            assert all((parameter is not fixed_reference for group in opt.param_groups for parameter in group['params']))
            history = [[0, math.log(2), 0.0]]
            for step in range(600):
                idx = torch.randint(len(ctx), (256,))
                policy_log_probs = logits.log_softmax(-1)
                c, a, b = (ctx[idx], chosen_id[idx], rejected_id[idx])
                loss = dpo_loss(policy_log_probs[c, a], policy_log_probs[c, b], fixed_reference[c, a], fixed_reference[c, b], beta)
                opt.zero_grad()
                loss.backward()
                opt.step()
                if (step + 1) % 30 == 0:
                    with torch.no_grad():
                        current = logits.log_softmax(-1)
                        history.append([step + 1, float(loss.detach()), float((current.exp() * (current - fixed_reference)).sum(-1).mean())])
            with torch.no_grad():
                lp = logits.log_softmax(-1)
                ratio = lp - fixed_reference
                acc = float((ratio[test_ctx, test_chosen] > ratio[test_ctx, test_rejected]).float().mean())
                kl = float((lp.exp() * (lp - fixed_reference)).sum(-1).mean())
                reward = float((lp.exp() * rewards).sum(-1).mean())
                base = float((reference_log_probs.exp() * rewards).sum(-1).mean())
                assert torch.equal(fixed_reference, reference_log_probs)
                records.append(dict(seed=seed, beta=beta, heldout_accuracy=acc, kl=kl, expected_reward=reward, reference_reward=base, probabilities=lp.exp().tolist()))
                traces.append(dict(seed=seed, beta=beta, history=history))
    return dict(initial_loss=initial, gradient=float(margin.grad), expected_gradient=float(expected_gradient), mask_check=True, reference_frozen=True, reference_excluded_from_optimizer=True, realised_reward_order_accuracy=realised_ceiling, expected_noise_ceiling=expected_ceiling, accuracy_standard_error=math.sqrt(realised_ceiling * (1 - realised_ceiling) / 4096), population_targets=targets, records=records, traces=traces, seconds=time.perf_counter() - start)

## Article training example

These cells define the article variables in notebook scope, so its training loop can be copied and modified directly. The following full comparison runs independently.

In [ ]:
# Recreate the article's named variables in notebook scope.
ctx, chosen_id, rejected_id, rewards = preference_data(120)
reference_log_probs = torch.log_softmax(torch.tensor(
    [[0.6 * np.cos(c + j * 0.8) for j in range(4)]
     for c in range(12)]), dim=-1)


In [ ]:
# This is the article's trainable policy and fixed reference example.
torch.manual_seed(0)
logits = nn.Parameter(reference_log_probs.clone())
fixed_reference = reference_log_probs.detach().clone()
opt = torch.optim.Adam([logits], lr=0.03)
assert all(parameter is not fixed_reference
           for group in opt.param_groups for parameter in group['params'])
for step in range(600):
    idx = torch.randint(len(ctx), (256,))
    c, a, b = ctx[idx], chosen_id[idx], rejected_id[idx]
    logp = logits.log_softmax(-1)
    loss = dpo_loss(logp[c, a], logp[c, b],
                    fixed_reference[c, a], fixed_reference[c, b],
                    beta=0.5)
    opt.zero_grad()
    loss.backward()
    opt.step()
assert torch.equal(fixed_reference, reference_log_probs)
print('Example KL:', float((logits.log_softmax(-1).exp() *
      (logits.log_softmax(-1) - fixed_reference)).sum(-1).mean().detach()))


## Run and print the complete comparison

All measurements below are calculated in this notebook. Fixed seeds fix samples, while hardware and library changes can affect timing and floating-point details.

In [ ]:
result = dpo_experiment()
def print_table(result, topic):
    """Print every experimental row, followed by the article's grouped means."""
    print('beta accuracy kl reward population_kl')
    for beta in [0.1, 0.5, 1.0]:
        rows = [row for row in result['records'] if row['beta'] == beta]
        target = next((row['kl'] for row in result['population_targets'] if row['beta'] == beta))
        print(beta, *[np.mean([row[key] for row in rows]) for key in ['heldout_accuracy', 'kl', 'expected_reward']], target)
    print('Expected noise ceiling:', result['expected_noise_ceiling'])
    print('True-reward ordering on realised test labels:', result['realised_reward_order_accuracy'])
    print('Nonzero-margin gradient:', result['gradient'], 'Expected:', result['expected_gradient'])
print_table(result, 'dpo')

## Dpo plots

The figure uses the newly computed result, with baselines and units labelled.

In [ ]:
def dpo_plots(result):
    figure, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    for index, beta in enumerate([.1, .5, 1.]):
        rows = [row for row in result['records'] if row['beta'] == beta]
        positions = beta + np.array([-.012, 0, .012])
        for axis, key in zip(axes, ['heldout_accuracy', 'kl', 'expected_reward']):
            axis.scatter(positions, [row[key] for row in rows], s=45, label=f'β={beta}')
    axes[0].axhline(result['expected_noise_ceiling'], linestyle='--', color='grey', label='Expected label-noise ceiling')
    axes[0].axhline(result['realised_reward_order_accuracy'], linestyle=':', color='black', label='True-reward ordering on test labels')
    axes[0].set(ylabel='Log-ratio preference accuracy', ylim=(.68, .75))
    axes[1].scatter([row['beta'] for row in result['population_targets']], [row['kl'] for row in result['population_targets']], marker='x', color='black', s=65, label='Population optimum')
    axes[1].set(ylabel='KL(policy || reference), nats')
    axes[2].set(ylabel='Expected synthetic reward')
    for axis in axes:
        axis.set(xlabel='Beta', xticks=[.1, .5, 1.])
    axes[0].legend(fontsize=8, loc='lower left')
    axes[1].legend(fontsize=9)
    figure.tight_layout()
    return figure
dpo_plots(result)
plt.show()

## Animated experiment

The fitted seed-zero KL trajectories are compared with the population-optimal policies and the fixed reference. Similar implied-reward ordering across beta is expected for this unrestricted table.

In [ ]:
def progress_animation(result, topic):
    figure, axes = plt.subplots(figsize=(8, 4.5))
    frames = list(range(15)) + [14] * 6

    def update(frame):
        fraction = (frame + 1) / 15
        axis = axes
        axis.clear()
        for trace in result['traces']:
            if trace['seed'] != 0:
                continue
            history = trace['history']
            count = max(1, int(fraction * len(history)))
            rows = history[:count]
            line, = axis.plot([row[0] for row in rows], [row[2] for row in rows], label=f"Fitted β={trace['beta']}")
            target = next((row['kl'] for row in result['population_targets'] if row['beta'] == trace['beta']))
            axis.axhline(target, color=line.get_color(), linestyle='--', alpha=0.5)
        axis.axhline(0, color='grey', linestyle=':', label='Fixed reference (KL=0)')
        axis.set(xlim=(-15, 615), ylim=(-0.05, 1.65), xlabel='Adam update', ylabel='KL(policy || reference), nats', title='Seed 0; dashed lines: population targets')
        axis.legend(fontsize=9)
        figure.tight_layout()
        return []
    return FuncAnimation(figure, update, frames=frames, interval=333, blit=False)
from IPython.display import Image, display
animation = progress_animation(result, 'dpo')
animation.save('dpo_progress.gif', writer=PillowWriter(fps=3))
plt.close(animation._fig)
display(Image(filename='dpo_progress.gif'))

## Exercises and limits

Compare fitted KL with population targets. Change the reference distribution and distinguish raw policy ordering from implied-reward ordering. Replace one noisy pair with deterministic labels and observe the unbounded margin; finite beta alone does not guarantee a finite optimum.

## References

DPO: https://arxiv.org/abs/2305.18290; Preference-learning caveats: https://arxiv.org/abs/2310.12036